In [2]:
# --- Importar todas las librerías necesarias ---
import pandas as pd
from collections import Counter
import re
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, recall_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuración de Archivos de Entrada ---
# Asegúrate de que todos estos archivos estén en la misma carpeta que tu notebook
archivo_j48_raw = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\j48/variables_fusion_j48"
archivo_ibk_raw = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\ibk/variables_fusion_ibk"
ruta_entrenamiento_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/combined_training.csv"
ruta_prueba_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_test.csv"
ruta_externa_completo = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion\Combinado/fusion_external.csv"

print("✅ Librerías y configuración cargadas.")

✅ Librerías y configuración cargadas.


In [3]:
# --- Fase 1: Encontrar Atributos de Consenso ---
print("--- Iniciando Fase 1: Búsqueda de Consenso ---")

# Usamos el CSV grande como un "diccionario" para traducir los índices a nombres
df_headers = pd.read_csv(ruta_entrenamiento_completo, sep=',', nrows=0)
lista_nombres_atributos = df_headers.columns.tolist()

# Combinamos los resultados numéricos de J48 e IBk
todos_los_indices = []
for archivo_raw in [archivo_j48_raw, archivo_ibk_raw]:
    with open(archivo_raw, 'r') as f:
        for line in f:
            numeros_encontrados = re.findall(r'\d+', line)
            todos_los_indices.extend([int(num) for num in numeros_encontrados])

frecuencia_indices = Counter(todos_los_indices)

# Filtramos para quedarnos solo con los que tienen frecuencia 2 o más
atributos_consenso = []
num_atributos_reales = len(lista_nombres_atributos) - 1
for indice, frecuencia in frecuencia_indices.items():
    if frecuencia >= 2 and 0 < indice <= num_atributos_reales:
        atributos_consenso.append(lista_nombres_atributos[indice - 1])

print(f"\nSe encontraron {len(Counter(todos_los_indices))} atributos únicos en total.")
print(f"✅ Fase 1 completada: Se encontraron {len(atributos_consenso)} atributos de consenso (frecuencia >= 2).")
print("\nAtributos de Consenso Seleccionados:")
print(atributos_consenso)

--- Iniciando Fase 1: Búsqueda de Consenso ---

Se encontraron 500 atributos únicos en total.
✅ Fase 1 completada: Se encontraron 92 atributos de consenso (frecuencia >= 2).

Atributos de Consenso Seleccionados:
['MW', 'F04[C-F]', 'F06[N-N]', 'F06[O-O]', 'piPC05', 'Yindex', 'F09[C-S]', 'F09[N-N]', 'F09[N-O]', 'CIC1', 'F10[C-F]', 'F10[N-N]', 'SM5_X', 'MDEC-44', 'SM1_Dz(p)', 'phLevel1', 'phLevel3', 's1_size', 's1_phSize', 's3_phSize', 'MATS1p', 'MATS2p', 'MATS7p', 'MATS5i', 'GATS5m', 'GATS6m', 'MACCSFP17', 'MACCSFP25', 'GATS1i', 'SpMax3_Bh(m)', 'MACCSFP39', 'MACCSFP42', 'MACCSFP54', 'P_VSA_m_4', 'P_VSA_p_3', 'MACCSFP59', 'MACCSFP66', 'MACCSFP73', 'Eta_beta_A', 'Eig06_EA(dm)', 'Eig15_EA(dm)', 'Eig12_AEA(dm)', 'Eig15_AEA(dm)', 'MACCSFP131', 'MACCSFP132', 'MACCSFP134', 'MACCSFP135', 'MACCSFP151', 'MACCSFP155', 'C-019', 'MACCSFP159', 'MACCSFP160', 'MACCSFP163', 'C-041', 'H-046', 'SsssCH', 'SdssC', 'SssssN+', 'CATS2D_07_DD', 'CATS2D_05_DA', 'CATS2D_03_DL', 'CATS2D_08_DL', 'CATS2D_06_AA', 'CAT

In [4]:
# --- Fase 2: Crear Datasets de Consenso ---
print("--- Iniciando Fase 2: Creación de Datasets ---")

# Tu lista de 92 atributos de consenso que encontraste en la Celda 1
atributos_consenso_raw = [
    'MW', 'F04[C-F]', 'F06[N-N]', 'F06[O-O]', 'piPC05', 'Yindex', 'F09[C-S]', 'F09[N-N]', 'F09[N-O]', 
    'CIC1', 'F10[C-F]', 'F10[N-N]', 'SM5_X', 'MDEC-44', 'SM1_Dz(p)', 'phLevel1', 'phLevel3', 's1_size', 
    's1_phSize', 's3_phSize', 'MATS1p', 'MATS2p', 'MATS7p', 'MATS5i', 'GATS5m', 'GATS6m', 'MACCSFP17', 
    'MACCSFP25', 'GATS1i', 'SpMax3_Bh(m)', 'MACCSFP39', 'MACCSFP42', 'MACCSFP54', 'P_VSA_m_4', 
    'P_VSA_p_3', 'MACCSFP59', 'MACCSFP66', 'MACCSFP73', 'Eta_beta_A', 'Eig06_EA(dm)', 'Eig15_EA(dm)', 
    'Eig12_AEA(dm)', 'Eig15_AEA(dm)', 'MACCSFP131', 'MACCSFP132', 'MACCSFP134', 'MACCSFP135', 'MACCSFP151', 
    'MACCSFP155', 'C-019', 'MACCSFP159', 'MACCSFP160', 'MACCSFP163', 'C-041', 'H-046', 'SsssCH', 'SdssC', 
    'SssssN+', 'CATS2D_07_DD', 'CATS2D_05_DA', 'CATS2D_03_DL', 'CATS2D_08_DL', 'CATS2D_06_AA', 
    'CATS2D_05_AP', 'CATS2D_07_AP', 'CATS2D_05_AN', 'CATS2D_07_NL', 'CATS2D_00_LL', 'CATS2D_03_LL', 
    'SHED_AL', 'SHED_LL', 'B01[O-S]', 'B04[N-N]', 'B04[O-S]', 'B07[C-S]', 'B08[C-C]', 'B08[C-S]', 
    'F01[N-N]', 'AMW', 'F03[N-S]', 'Psi_e_0', 'PCR', 'MACCSFP18', 'MACCSFP65', 'Eig13_EA(dm)', 
    'B06[C-C]', 'B07[C-C]', 'Mp', 'Psi_e_A', 'stdMW', 'stdP', 'LOGPcons'
]

# ¡ACCIÓN CLAVE! Excluimos LOGPcons para garantizar la validez científica del modelo
if 'LOGPcons' in atributos_consenso_raw:
    print("Confirmado: 'LOGPcons' encontrado en la lista. Será excluido.")
    atributos_consenso_final = [attr for attr in atributos_consenso_raw if attr != 'LOGPcons']
else:
    atributos_consenso_final = atributos_consenso_raw

# Cargamos los datasets completos
df_train_full = pd.read_csv(ruta_entrenamiento_completo, sep=',')
df_test_full = pd.read_csv(ruta_prueba_completo, sep=',')
df_external_full = pd.read_csv(ruta_externa_completo, sep=',')

columna_clase = df_train_full.columns[-1]
columnas_finales = atributos_consenso_final + [columna_clase]

# Creamos los dataframes reducidos
df_train = df_train_full[columnas_finales]
df_test = df_test_full[columnas_finales]
df_external = df_external_full[columnas_finales]

# Separamos en X (características) e y (clase)
X_train, y_train = df_train[atributos_consenso_final], df_train[columna_clase]
X_test, y_test = df_test[atributos_consenso_final], df_test[columna_clase]
X_external, y_external = df_external[atributos_consenso_final], df_external[columna_clase]

print(f"\n✅ Fase 2 completada. Datasets de consenso creados con {X_train.shape[1]} atributos finales.")

# Generamos una tabla de previsualización para verificar
print("\n--- Vista Previa de la Tabla de Datos de Entrenamiento Final ---")
display(df_train.head())

--- Iniciando Fase 2: Creación de Datasets ---
Confirmado: 'LOGPcons' encontrado en la lista. Será excluido.

✅ Fase 2 completada. Datasets de consenso creados con 91 atributos finales.

--- Vista Previa de la Tabla de Datos de Entrenamiento Final ---


,MW,F04[C-F],F06[N-N],F06[O-O],piPC05,Yindex,F09[C-S],F09[N-N],F09[N-O],CIC1,...,MACCSFP18,MACCSFP65,Eig13_EA(dm),B06[C-C],B07[C-C],Mp,Psi_e_A,stdMW,stdP,Actividad
0,393.566,0,1,0,6.0519,0.4491,0,1,0,2.1854,...,0,1,0.0000,1,1,0.6771,7.5345,5.5893,0.2949,Act1
1,406.734,0,0,0,4.9558,0.5939,0,0,0,3.4819,...,0,0,0.0000,1,1,0.6118,7.4397,5.5320,0.2997,Act1
2,422.734,0,0,0,4.9558,0.5784,0,0,0,3.3917,...,0,0,0.0000,1,1,0.6097,7.5611,5.6261,0.2982,Act-1
3,424.704,0,0,0,4.9628,0.5826,0,0,0,3.1742,...,0,0,0.0000,1,1,0.6084,7.6389,5.7287,0.2969,Act-1
4,2181.604,0,19,11,6.5697,0.2194,0,18,31,4.5505,...,0,0,1.9914,1,1,0.6162,8.3892,6.1917,0.2804,Act-1


In [5]:
# --- Fase 3: Evaluación Comparativa ---
print("\n--- Iniciando Fase 3: Evaluación Comparativa de Modelos ---")

modelos_a_evaluar = {
    "SVM": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Árbol de Decisión (J48)": DecisionTreeClassifier(random_state=42),
    "k-NN (k=5)": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

resultados_comparativos = {}
print("Resultados (ROC AUC en Prueba Interna / Prueba Externa):\n" + "="*60)

for nombre_modelo, modelo in modelos_a_evaluar.items():
    modelo.fit(X_train, y_train)
    
    # Evaluar en Test Set para decidir el campeón
    y_proba_test = modelo.predict_proba(X_test)[:, 1]
    roc_auc_test = roc_auc_score(y_test, y_proba_test)
    resultados_comparativos[nombre_modelo] = roc_auc_test

    # Evaluar en External Set para el reporte
    y_proba_ext = modelo.predict_proba(X_external)[:, 1]
    roc_auc_ext = roc_auc_score(y_external, y_proba_ext)
    
    print(f"  > {nombre_modelo:<25} | {roc_auc_test:.4f} / {roc_auc_ext:.4f}")

campeon_nombre = max(resultados_comparativos, key=resultados_comparativos.get)
print("\n" + "="*60)
print(f"✅ Fase 3 completada. El modelo campeón es: {campeon_nombre} (ROC AUC: {resultados_comparativos[campeon_nombre]:.4f})")


--- Iniciando Fase 3: Evaluación Comparativa de Modelos ---
Resultados (ROC AUC en Prueba Interna / Prueba Externa):
  > SVM                       | 0.7504 / 0.6007
  > Random Forest             | 0.8216 / 0.6228
  > Árbol de Decisión (J48)   | 0.7098 / 0.6277
  > k-NN (k=5)                | 0.7232 / 0.5581
  > Naive Bayes               | 0.7676 / 0.6686

✅ Fase 3 completada. El modelo campeón es: Random Forest (ROC AUC: 0.8216)


In [6]:
# --- Fase 4: Optimización del Campeón ---
print(f"\n--- Iniciando Fase 4: Optimizando al Campeón ({campeon_nombre}) ---")

# Definimos los parámetros a probar para los modelos más comunes
param_grids = {
    "Random Forest": {'n_estimators': [100, 200, 300], 'max_features': ['sqrt', 'log2'], 'max_depth': [10, 20, None]},
    "SVM": {'C': [1, 10, 100], 'gamma': ['scale', 'auto'], 'kernel': ['rbf']}
}

# Elegimos el grid de parámetros para nuestro campeón
param_grid_campeon = param_grids.get(campeon_nombre)

if param_grid_campeon:
    # Usamos el modelo con random_state para consistencia
    modelo_campeon_base = modelos_a_evaluar[campeon_nombre]
    
    grid_search = GridSearchCV(modelo_campeon_base, param_grid_campeon, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)
    
    modelo_final_optimizado = grid_search.best_estimator_
    print(f"\nMejores parámetros encontrados: {grid_search.best_params_}")
    
    print("\n" + "="*60 + "\n--- REPORTE FINAL DEL MODELO OPTIMIZADO ---\n" + "="*60)
    
    for nombre_set, X_eval, y_eval in [("Prueba Interna", X_test, y_test), ("Prueba Externa", X_external, y_external)]:
        y_pred = modelo_final_optimizado.predict(X_eval)
        y_proba = modelo_final_optimizado.predict_proba(X_eval)[:, 1]
        
        print(f"\nResultados en: {nombre_set}")
        print("-" * 30)
        print(f"  ROC AUC:          {roc_auc_score(y_eval, y_proba):.4f}")
        print(f"  BACC:             {balanced_accuracy_score(y_eval, y_pred):.4f}")
        print(f"  Sensitivity:      {recall_score(y_eval, y_pred, pos_label='Act1'):.4f}")
        print(f"  Specificity:      {recall_score(y_eval, y_pred, pos_label='Act-1'):.4f}")

else:
    print(f"\nNo se ha definido un grid de optimización para '{campeon_nombre}'. El proceso termina aquí.")

print("\n\n✅ ¡PROYECTO COMPLETADO!")


--- Iniciando Fase 4: Optimizando al Campeón (Random Forest) ---
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Mejores parámetros encontrados: {'max_depth': 10, 'max_features': 'log2', 'n_estimators': 200}

--- REPORTE FINAL DEL MODELO OPTIMIZADO ---

Resultados en: Prueba Interna
------------------------------
  ROC AUC:          0.8296
  BACC:             0.7306
  Sensitivity:      0.8182
  Specificity:      0.6430

Resultados en: Prueba Externa
------------------------------
  ROC AUC:          0.6274
  BACC:             0.5851
  Sensitivity:      0.9842
  Specificity:      0.1860


✅ ¡PROYECTO COMPLETADO!
